# 1 Generate Noisy Data and Fitting

This is an optional notebook that is used to generate noisy data for the manuscript. We note that we provide the output of this notebook directly as a download. 

This notebook is used to generate and fit all the data required for the paper. We will generate data for the following noise cases: 

1, 2, 3, 4, 5, 6, 7, where each case corresponds to a different noise level. 

## Imports

In [1]:
%load_ext autoreload
%autoreload 2

from m3util.util.IO import download_and_unzip
from m3util.util.h5 import find_groups_with_string
from belearn.dataset.dataset import BE_Dataset


import numpy as np


## Loading data for SHO fitting


In [2]:
# Download the data file from Zenodo
url = 'https://zenodo.org/record/7774788/files/PZT_2080_raw_data.h5?download=1'

# Specify the filename and the path to save the file
filename = '/data_raw5.h5'
save_path = './Data'

# download the file
download_and_unzip(filename, url, save_path)

downloading data
...100%, 1663 MB, 3602 KB/s, 472 seconds passed

In [3]:
data_path = save_path + '/' + filename

# instantiate the dataset object
dataset = BE_Dataset(data_path)

# print the contents of the file
dataset.print_be_tree

/
├ Measurement_000
  ---------------
  ├ Channel_000
    -----------
    ├ Bin_FFT
    ├ Bin_Frequencies
    ├ Bin_Indices
    ├ Bin_Step
    ├ Bin_Wfm_Type
    ├ Excitation_Waveform
    ├ Noise_Floor
    ├ Position_Indices
    ├ Position_Values
    ├ Raw_Data
    ├ Spatially_Averaged_Plot_Group_000
      ---------------------------------
      ├ Bin_Frequencies
      ├ Max_Response
      ├ Mean_Spectrogram
      ├ Min_Response
      ├ Spectroscopic_Parameter
      ├ Step_Averaged_Response
    ├ Spatially_Averaged_Plot_Group_001
      ---------------------------------
      ├ Bin_Frequencies
      ├ Max_Response
      ├ Mean_Spectrogram
      ├ Min_Response
      ├ Spectroscopic_Parameter
      ├ Step_Averaged_Response
    ├ Spectroscopic_Indices
    ├ Spectroscopic_Values
    ├ UDVS
    ├ UDVS_Indices
Datasets and datagroups within the file:
------------------------------------
/
/Measurement_000
/Measurement_000/Channel_000
/Measurement_000/Channel_000/Bin_FFT
/Measurement_000/Chann

## Generates Noisy Data

This function will generate noisy records and save them as an h5_main file in the USID format. This allows the data to be computed with the Pycroscopy SHO Fitter. 

In [4]:
# calculates the standard deviation and uses that for the noise
noise_STD = np.std(dataset.raw_SHO_data)

# prints the standard deviation
print(noise_STD)

0.0038833667


In [5]:
dataset.raw_SHO_data.shape

(3600, 63360)

In [8]:
dataset.generate_noisy_data_records(noise_levels = np.arange(1,9), 
                                    verbose=True, 
                                    noise_STD=noise_STD)

Noise standard deviation: 0.0038833667058497667
The STD of the data is: 0.0038833667058497667
Adding noise level 1
Adding noise level 2
Adding noise level 3
Adding noise level 4
Adding noise level 5
Adding noise level 6
Adding noise level 7
Adding noise level 8


## SHO fits on all the datasets

This will take some time, Each fit takes about 10 minutes to complete. 

In [54]:
from belearn.dataset.fitters.sho import SHOFitter

In [55]:
shofitter = SHOFitter(dataset)

In [57]:
out = [f"Noisy_Data_{i}" for i in np.arange(1,9)]
out.append("Raw_Data")

for data in out:
    if find_groups_with_string(dataset.file, data) == []:
        print(f"Fitting {data}")
        shofitter.fit(dataset = data, h5_sho_targ_grp = f"{data}_SHO_Fit", max_mem=1024*64, max_cores= 20)
    else:
        print(f"{data} already fit. Skipping...")

Noisy_Data_1 already fit. Skipping...
Noisy_Data_2 already fit. Skipping...
Noisy_Data_3 already fit. Skipping...
Noisy_Data_4 already fit. Skipping...
Noisy_Data_5 already fit. Skipping...
Noisy_Data_6 already fit. Skipping...
Noisy_Data_7 already fit. Skipping...
Noisy_Data_8 already fit. Skipping...
Raw_Data already fit. Skipping...


In [58]:
out = [f"Noisy_Data_{i}" for i in np.arange(1,9)]
out.append("Raw_Data")

for data in out:
    print(f"Fitting {data}")
    shofitter.fit(dataset = data, h5_sho_targ_grp = f"{data}_SHO_Fit", max_mem=1024*64, max_cores= 20)

Fitting Noisy_Data_1
Working on:
./Data//data_raw4.h5
['Y', 'X'] [60, 60]


SHO Fits will be written to:
./Data/data_raw4.h5


Could not add group - it might already exist.
Consider calling test() to check results before calling compute() which computes on the entire dataset and writes results to the HDF5 file

Note: SHO_Fit has already been performed PARTIALLY with the same parameters. compute() will resuming computation in the last group below. To choose a different group call use_patial_computation()Set override to True to force fresh computation or resume from a data group besides the last in the list.

[<HDF5 group "/Measurement_000/Channel_000/Noisy_Data_1-SHO_Fit_000" (7 members)>]
SHO fits for Noisy_Data_1 already exist. Skipping....
Fitting Noisy_Data_2
Working on:
./Data//data_raw4.h5
['Y', 'X'] [60, 60]


SHO Fits will be written to:
./Data/data_raw4.h5


Could not add group - it might already exist.
Consider calling test() to check results before calling compute() which com

### Checks the results to make sure it was saved correctly

In [14]:
# print the contents of the file
dataset.print_be_tree

/
├ Measurement_000
  ---------------
  ├ Channel_000
    -----------
    ├ Bin_FFT
    ├ Bin_Frequencies
    ├ Bin_Indices
    ├ Bin_Step
    ├ Bin_Wfm_Type
    ├ Excitation_Waveform
    ├ Noise_Floor
    ├ Noisy_Data_1
    ├ Noisy_Data_1-SHO_Fit_000
      ------------------------
      ├ Fit
      ├ Guess
      ├ Spectroscopic_Indices
      ├ Spectroscopic_Values
      ├ completed_fit_positions
      ├ completed_guess_positions
      ├ completed_positions
    ├ Noisy_Data_2
    ├ Noisy_Data_3
    ├ Noisy_Data_4
    ├ Noisy_Data_5
    ├ Noisy_Data_6
    ├ Noisy_Data_7
    ├ Noisy_Data_8
    ├ Position_Indices
    ├ Position_Values
    ├ Raw_Data
    ├ Spatially_Averaged_Plot_Group_000
      ---------------------------------
      ├ Bin_Frequencies
      ├ Max_Response
      ├ Mean_Spectrogram
      ├ Min_Response
      ├ Spectroscopic_Parameter
      ├ Step_Averaged_Response
    ├ Spatially_Averaged_Plot_Group_001
      ---------------------------------
      ├ Bin_Frequencies
      ├